# 03 – Iteration 3: Feature Selection (with 5% Stratified Sampling)

This notebook performs feature selection using:

1. **Feature table** from Notebook 2  
2. **5% stratified sampling** by `y_anom` to reduce compute cost while maintaining precision  
3. **Feature selection pipeline** from `src/selection.py`:
   - Missingness filtering
   - Zero-variance filtering
   - Correlation pruning
   - RandomForest-based ranking  
4. Saves:
   - `selected_features_iter3.txt`
   - `feature_rankings_iter3.csv`

This step prepares the reduced set of features to be used in:
- Notebook 4 (Dataset Preparation)
- Notebook 5 (Model Training)


Imports & paths

In [1]:
import os
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

# Import selection utilities
SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from features import get_feature_columns
from selection import select_features

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

FEATURES_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "features"))
SELECTED_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results", "selected"))
os.makedirs(SELECTED_DIR, exist_ok=True)

FEATURES_PARQUET_PATH = os.path.join(FEATURES_DIR, "features_dataset2_iter3.parquet")

print("FEATURES_PARQUET_PATH:", FEATURES_PARQUET_PATH)
print("SELECTED_DIR:", SELECTED_DIR)


FEATURES_PARQUET_PATH: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\features\features_dataset2_iter3.parquet
SELECTED_DIR: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\selected


Load features & define ID / label

In [2]:
feat_df = pd.read_parquet(FEATURES_PARQUET_PATH)

print("Feature table loaded.")
print("Shape:", feat_df.shape)
feat_df.head()

Feature table loaded.
Shape: (21195970, 19)


,POLISSA_SUBM,CODI_ANOMALIA,START_DATE,END_DATE,US_AIGUA_SUBM,SECCIO_CENSAL,NUMEROSERIECONTADOR,CONSUMO_REAL,FECHA_HORA,flag_anom_32768,flag_anom_163840,y_anom,datetime,cons_lag1,delta1,meter_mean,meter_std,cons_z_meter,period_hours
0,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 00:36:20,False,True,1,2024-01-01 00:36:20,NaN,NaN,0.0,0.0,NaN,1488.0
1,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 01:36:20,False,True,1,2024-01-01 01:36:20,0.0,0.0,0.0,0.0,NaN,1488.0
2,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 02:36:32,False,True,1,2024-01-01 02:36:32,0.0,0.0,0.0,0.0,NaN,1488.0
3,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 03:36:32,False,True,1,2024-01-01 03:36:32,0.0,0.0,0.0,0.0,NaN,1488.0
4,UXOZJTJWD74S44CF,163840,2023-09-07,2023-11-08,DOMÈSTIC,0801906022,01172604,0.0,2024-01-01 04:36:32,False,True,1,2024-01-01 04:36:32,0.0,0.0,0.0,0.0,NaN,1488.0


In [3]:
ID_COL = "NUMEROSERIECONTADOR"
LABEL_COL = "y_anom"

assert ID_COL in feat_df.columns
assert LABEL_COL in feat_df.columns

5% Stratified Sample

In [4]:
# We stratify on the anomaly label to preserve ratios
sample_frac = 0.05   # 5%

df_sample, _ = train_test_split(
    feat_df,
    train_size=sample_frac,
    stratify=feat_df[LABEL_COL],
    random_state=42,
)

print("5% sample shape:", df_sample.shape)
df_sample[LABEL_COL].value_counts(normalize=True)


5% sample shape: (1059798, 19)


y_anom
1    0.999493
0    0.000507
Name: proportion, dtype: float64

Identify Numeric Candidate Features

In [5]:
candidate_features = get_feature_columns(
    df_sample,
    id_col=ID_COL,
    label_col=LABEL_COL,
)

print(f"Candidate numeric features: {len(candidate_features)}")
candidate_features[:15]


Candidate numeric features: 10


['CODI_ANOMALIA',
 'CONSUMO_REAL',
 'flag_anom_32768',
 'flag_anom_163840',
 'cons_lag1',
 'delta1',
 'meter_mean',
 'meter_std',
 'cons_z_meter',
 'period_hours']

Feature Selection Pipeline

In [6]:
MAX_MISSING_RATIO = 0.40   # drop features with >40% NA
MAX_CORR = 0.95            # correlation threshold
TOP_K = 50                 # keep top 50 features

selected_features, rankings = select_features(
    df=df_sample,
    feature_cols=candidate_features,
    label_col=LABEL_COL,
    max_missing_ratio=MAX_MISSING_RATIO,
    max_corr=MAX_CORR,
    top_k=TOP_K,
)

print("\nSelected features:", len(selected_features))
selected_features[:20]



Selected features: 10


['CODI_ANOMALIA',
 'period_hours',
 'meter_std',
 'meter_mean',
 'flag_anom_163840',
 'cons_z_meter',
 'flag_anom_32768',
 'CONSUMO_REAL',
 'cons_lag1',
 'delta1']

Save Selected Features & Rankings

In [7]:
SELECTED_TXT_PATH = os.path.join(SELECTED_DIR, "selected_features_iter3.txt")
RANKINGS_CSV_PATH = os.path.join(SELECTED_DIR, "feature_rankings_iter3.csv")

# Save selected features list
with open(SELECTED_TXT_PATH, "w") as f:
    for feat in selected_features:
        f.write(feat + "\n")

# Save full ranking
rankings.to_csv(RANKINGS_CSV_PATH, index=False)

print("\n[ok] Saved selected features to:", SELECTED_TXT_PATH)
print("[ok] Saved feature rankings to :", RANKINGS_CSV_PATH)



[ok] Saved selected features to: c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\selected\selected_features_iter3.txt
[ok] Saved feature rankings to : c:\Users\joan\Desktop\FEINA\UPF\Course\Fourth_year\Primer_Trimestre\Project_Management\AB\AB_DataChallenge.Team102D\iteration_3\results\selected\feature_rankings_iter3.csv


Summary for Documentation

In [8]:
print("Total numeric candidate features:", len(candidate_features))
print("Selected feature count:", len(selected_features))

print("\nTop 10 Selected Features by Importance:")
display(rankings[rankings["feature"].isin(selected_features)].head(10))

Total numeric candidate features: 10
Selected feature count: 10

Top 10 Selected Features by Importance:


,feature,importance,mean,std,auc_single
0,CODI_ANOMALIA,0.539457,191745.400790,284278.238534,1.000000
9,period_hours,0.178197,3634.074428,68430.178576,0.229806
7,meter_std,0.086871,458.491848,11397.547765,NaN
6,meter_mean,0.069017,39.092365,1351.283640,NaN
3,flag_anom_163840,0.054627,0.460679,0.498452,0.730456
8,cons_z_meter,0.045356,0.000361,0.963687,NaN
2,flag_anom_32768,0.021290,0.319431,0.466256,0.659796
1,CONSUMO_REAL,0.003889,11.308989,196.500396,NaN
4,cons_lag1,0.000813,11.296914,118.526575,NaN
5,delta1,0.000482,-2.242582,113.630067,NaN
